You can download the `requirements.txt` for this course from the workspace of this lab. `File --> Open...`

# L2: Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

The libraries are already installed in the classroom. If you're running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [2]:
from crewai import Agent, Task, Crew

- As a LLM for your agents, you'll be using OpenAI's `gpt-3.5-turbo`.

**Optional Note:** crewAI also allow other popular models to be used as a LLM for your Agents. You can see some of the examples at the [bottom of the notebook](#1).

In [ ]:
import os
from com.example.agentic.utils import get_openai_api_key

openai_api_key = get_openai_api_key()
os.environ["OPENAI_MODEL_NAME"] = 'gemma4:latest'
#os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

from com.example.agentic.agent.LLMManager import LLMManager
llm = LLMManager.create_llm('openai')
llm.call('Hello')

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner

**Note**: The benefit of using _multiple strings_ :
```Python
varname = "line 1 of text"
          "line 2 of text"
```

versus the _triple quote docstring_:
```Python
varname = """line 1 of text
             line 2 of text
          """
```
is that it can avoid adding those whitespaces and newline characters, making it better formatted to be passed to the LLM.

In [3]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True
)

### Agent: Writer

In [4]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True
)

### Agent: Editor

In [5]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [6]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [7]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [8]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution. 

In [9]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=True,
    #llm=llm
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see in the video.

In [10]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 9a854b83-3c27-4d4f-b782-df5cb896f92a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.            │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│  ID: fdfbf105-a1ae-4874-b1df-b5376daaa9f9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.            │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Comprehensive Content Plan Document**                                                                        │
│                                                                                                                 │
│  **I. Title and Tagline**                                                                                       │
│  Title: "Empowering AI Adoption: Trends, Players, and Opportunities"                                            │
│  Tagline: "Unlocking the Potential of Artificial Intelligence in [Industry/Market]"                             │
│  AI Adoption Enabling Future Insights                                                                           │
│                                                                                                                 │
│  **II. Overview and Introduction**                                                                              │
│                                                                                                                 │
│  The integration of Artificial Intelligence (AI) has transformed various industries, including healthcare,      │
│  finance, education, and transportation, into more efficient and innovative sectors. As AI adoption is          │
│  increasing rapidly, it's essential to understand the latest trends, key players, and noteworthy news related   │
│  to this technology.                                                                                            │
│                                                                                                                 │
│  **III. Priority List - Latest Trends, Key Players, and Noteworthy News on Artificial Intelligence**            │
│                                                                                                                 │
│  The top emerging trends in AI include:                                                                         │
│  - **Edge AI**: Advancements in edge computing enable real-time processing at the device level.                 │
│  - **Explainable AI (XAI)**: Ensuring accountability by visualizing decision-making processes.                  │
│  - **Artificial General Intelligence (AGI)**: The possibility of a superintelligent AI surpassing human         │
│  intelligence.                                                                                                  │
│                                                                                                                 │
│  Key Players:                                                                                                   │
│  - Google's AI Research                                                                                         │
│  - Microsoft's AI Health Initiative                                                                             │
│  - IBM Watson                                                                                                   │
│                                                                                                                 │
│  Noteworthy News and Announcements:                                                                             │
│                                                                                                                 │
│  - "Google Unveils New AI-Powered Service to Help Diagn

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.            │
│  2. Identify the target audience, considering their interests and pain points.                                  │
│  3. Develop a detailed content outline including an introduction, key points, and a call to action.             │
│  4. Include SEO keywords and relevant data or sources.                                                          │
│  Agent: Content Planner                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Task: Proofread the given blog post for grammatical errors and alignment with the brand's voice.               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Empowering AI Adoption: Trends, Players, and Opportunities**                                                 │
│  ======================================================================                                         │
│                                                                                                                 │
│  **I. Introduction**                                                                                            │
│  -----------------                                                                                              │
│                                                                                                                 │
│  The integration of Artificial Intelligence (AI) has revolutionized various industries, including healthcare,   │
│  finance, education, and transportation. As AI adoption continues to increase rapidly, it is essential to       │
│  understand the latest trends, key players, and noteworthy news related to this technology. In this blog post,  │
│  we will delve into the world of Artificial Intelligence, exploring its current state, notable developments,    │
│  and future prospects.                                                                                          │
│                                                                                                                 │
│  **II. Latest Trends and Key Players**                                                                          │
│  --------------------------------------                                                                         │
│                                                                                                                 │
│  ### Edge AI                                                                                                    │
│                                                                                                                 │
│  Advancements in edge computing enable real-time processing at the device level.                                │
│  Edge AI enables accurate predictions by reducing data transfer costs on the device level.                      │
│  This approach is particularly useful for applications such as smart homes and cities, where timely responses   │
│  are crucial. Recent breakthroughs have resulted in more efficient deployment of AI models.                     │
│                                                                                                                 │
│  ### Explainable AI (XAI)                                                                                       │
│                                                                                                                 │
│  Explainable AI aims to provide clear explanations for complex AI-driven decisions and predictions.             │
│  To achieve this goal, systems must prioritize transparency, interpretability, Accessibility.                   │
│  By emphasizing these three aspects of explainability, we can trust AI technology in our daily lives more.      │
│  Furthermore, organizations using XAI will improve upon their decision-making processes.                        │
│                                                                                                                 │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Proofread the given blog post for grammatical errors and alignment with the brand's voice.               │
│  Agent: Editor                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 9a854b83-3c27-4d4f-b782-df5cb896f92a                                                                       │
│  Final Output: **Empowering AI Adoption: Trends, Players, and Opportunities**                                   │
│  ======================================================================                                         │
│                                                                                                                 │
│  **I. Introduction**                                                                                            │
│  -----------------                                                                                              │
│                                                                                                                 │
│  The integration of Artificial Intelligence (AI) has revolutionized various industries, including healthcare,   │
│  finance, education, and transportation. As AI adoption continues to increase rapidly, it is essential to       │
│  understand the latest trends, key players, and noteworthy news related to this technology. In this blog post,  │
│  we will delve into the world of Artificial Intelligence, exploring its current state, notable developments,    │
│  and future prospects.                                                                                          │
│                                                                                                                 │
│  **II. Latest Trends and Key Players**                                                                          │
│  --------------------------------------                                                                         │
│                                                                                                                 │
│  ### Edge AI                                                                                                    │
│                                                                                                                 │
│  Advancements in edge computing enable real-time processing at the device level.                                │
│  Edge AI enables accurate predictions by reducing data transfer costs on the device level.                      │
│  This approach is particularly useful for applications such as smart homes and cities, where timely responses   │
│  are crucial. Recent breakthroughs have resulted in more efficient deployment of AI models.                     │
│                                                                                                                 │
│  ### Explainable AI (XAI)                                                                                       │
│                                                                                                                 │
│  Explainable AI aims to provide clear explanations for complex AI-driven decisions and predictions.             │
│  To achieve this goal, systems must prioritize transparency, interpretability, Accessibility.                   │
│  By emphasizing these three aspects of explainability, we can trust AI technology in our daily lives more.      │
│  Furthermore, organizations using XAI will improve upon their decision-making processes.                        │
│                                                                                                                 │
│                                                       

╭────────────────────────── Trace Batch Finalization ──────────────────────────╮
│ ✅ Trace batch finalized with session ID:                                    │
│ e443bf78-2304-477c-99db-0d176fdf408d                                         │
│                                                                              │
│ 🔗 View here:                                                                │
│ https://app.crewai.com/crewai_plus/ephemeral_trace_batches/e443bf78-2304-477 │
│ c-99db-0d176fdf408d?access_code=TRACE-eba49d7d48                             │
│ 🔑 Access Code: TRACE-eba49d7d48                                             │
╰──────────────────────────────────────────────────────────────────────────────╯


- Display the results of your execution as markdown in the notebook.

In [11]:
from IPython.display import Markdown
Markdown(result.raw)

**Empowering AI Adoption: Trends, Players, and Opportunities**
======================================================================

**I. Introduction**
-----------------

The integration of Artificial Intelligence (AI) has revolutionized various industries, including healthcare, finance, education, and transportation. As AI adoption continues to increase rapidly, it is essential to understand the latest trends, key players, and noteworthy news related to this technology. In this blog post, we will delve into the world of Artificial Intelligence, exploring its current state, notable developments, and future prospects.

**II. Latest Trends and Key Players**
--------------------------------------

### Edge AI

Advancements in edge computing enable real-time processing at the device level.
Edge AI enables accurate predictions by reducing data transfer costs on the device level.
This approach is particularly useful for applications such as smart homes and cities, where timely responses are crucial. Recent breakthroughs have resulted in more efficient deployment of AI models.

### Explainable AI (XAI)

Explainable AI aims to provide clear explanations for complex AI-driven decisions and predictions.
To achieve this goal, systems must prioritize transparency, interpretability, Accessibility.
By emphasizing these three aspects of explainability, we can trust AI technology in our daily lives more. Furthermore, organizations using XAI will improve upon their decision-making processes.

 
**Artificial General Intelligence (AGI)**

Explainable AI's development was significantly aided by researchers specializing in AGI. Understanding theoretical limits requires a detailed analysis: Develop AGI systems based on prior knowledge of its challenges.
Understanding this intricate process is key to harnessing the incredible potential of AGI, though it might lead us into uncharted realms.

### Noteworthy News and Announcements

- **Google Unveils New AI-Powered Service to Help Diagnose Rare Diseases**

Last week, Google unveiled a new AI-powered service called "Google Health," designed to help diagnose rare diseases by analyzing medical images and genetic data.
This innovative tool promises improved accuracy and faster response times for patients requiring specialized treatment. Furthermore, investors believe Google's latest acquisition could bolster the development of cutting-edge treatments.

- **$31B Investment in Artificial Intelligence**

Microsoft has announced a $31 billion investment in its artificial intelligence (AI) research programs, further solidifying the company's commitment to AI development.
This significant update has brought forth opportunities for business leaders looking to expand the capabilities and range of their AI use.
To achieve full value from AI investments, it is necessary to ensure alignment between strategic objectives and specific application contexts.

### Notable Announcements

- **Alibaba Group Holding Limited**: Reports a significant increase in revenue from AI-driven business services
  
-  **Aerospace Industry Expansion:** Lockheed Martin reveals plans to develop advanced AI-powered systems for space exploration and satellite navigation.

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

In [ ]:
topic = "YOUR TOPIC HERE"
result = crew.kickoff(inputs={"topic": topic})

In [ ]:
Markdown(result)

<a name='1'></a>
 ## Other Popular Models as LLM for your Agents

#### Hugging Face (HuggingFaceHub endpoint)

```Python
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

### you will pass "llm" to your agent function
```

#### Mistral API

```Python
OPENAI_API_KEY=your-mistral-api-key
OPENAI_API_BASE=https://api.mistral.ai/v1
OPENAI_MODEL_NAME="mistral-small"
```

#### Cohere

```Python
from langchain_community.chat_models import ChatCohere
# Initialize language model
os.environ["COHERE_API_KEY"] = "your-cohere-api-key"
llm = ChatCohere()

### you will pass "llm" to your agent function
```

### For using Llama locally with Ollama and more, checkout the crewAI documentation on [Connecting to any LLM](https://docs.crewai.com/how-to/LLM-Connections/).